# 09 · Dimensiones, indicador y clasificación desde Gold

Este notebook reemplaza el contenido anterior. El recorrido es **Gold → D1 → D2 → D3 → indicador compuesto → clasificación → porcentajes**. El notebook 10 reúne los agregados y el 11 monitorea el cálculo.

Reinicie el kernel tras detener la ejecución anterior. Ejecute todas las celdas en orden. La primera ejecución del lote calcula desde Gold; repetir el mismo lote recupera checkpoints válidos. No se reconstruyen las capas anteriores ni se publica en AWS con esta configuración.


## Arquitectura de memoria aplicada

1. Preparación desde Gold y referencias persistentes en DuckDB.
2. Dask: **2 workers × 2 GiB, 1 hilo por worker**, cola de hasta 4 tareas y checkpoints por bloque. Se exigen al menos 6 GiB disponibles con esta configuración.
3. Los workers y la conexión de preparación se cierran; una conexión nueva integra D2 con **3.072 MB y 1 hilo** según `alertas_config.json`.
4. Se elige la prioridad por contrato sobre dos columnas y se incorpora la referencia completa mediante join. Se rechazan IDs duplicados.

Se conserva el lote y la caché de bloques. Reinicie el kernel antes de ejecutar para cargar el código nuevo. El cálculo puede usar disco temporal: los límites de DuckDB/Dask no representan toda la memoria del proceso. Para cambiar de máquina, revise la RAM disponible; no cambie el lote para resolver problemas de recursos.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'secop_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from flows.alertas_flow import alertas_flow, configuration
from secop_pipeline.alertas import GOLD

# Mantenga el lote para reanudar; un nombre nuevo obliga a recalcular desde Gold.
LOTE = 'desde_gold_v2_20261009'
PUBLICAR_AWS = False
config = configuration(ROOT/'alertas_config.json')
if config.version != 'alertas-cd-2.0':
    raise ValueError('Se requiere versión 2.0; revise configuración y reinicie el kernel.')
if not GOLD.is_file():
    raise FileNotFoundError(GOLD)
print('Fuente:', GOLD, '\nLote:', LOTE)
display(pd.Series(config.evidence(), name='Configuración').to_frame())


## 1. Qué calcula cada dimensión

| Dimensión | Medida | Interpretación |
|---|---|---|
| D1 · Concentración | Participación del proveedor en el valor contratado por la entidad en la ventana de referencia | Activa señal cuando supera el umbral y cumple mínimos de cobertura y tamaño |
| D2 · Valor atípico | Valor comparado con contratos históricos semejantes mediante logaritmo, mediana y MAD; alternativa P95 según el método registrado | Activa señal cuando supera la regla económica y hay historia suficiente |
| D3 · Acumulación de diciembre | Contratos de diciembre / contratos del año para la entidad, frente a años completos anteriores | Activa señal en contratos de diciembre si supera max(piso, Q3 + factor × IQR) y cumple mínimos |

Cada dimensión devuelve **1 = criterio de alerta activado, 0 = evaluado sin señal, NULL = no evaluable**. D3 es retrospectivo: se conoce al cerrar el año. Los parámetros efectivos están en la tabla anterior.

## 2. Calcular dimensiones y construir el indicador

Prefect ejecuta cada dimensión como una tarea separada, registra estadísticas y checkpoints, y luego combina por `id_contrato`. Valida que los joins no pierdan ni multipliquen contratos. Observe las tareas y el clúster desde el notebook 11 en otro kernel.


In [ ]:
# Celda lista para ejecutar después de reiniciar el kernel.
# Mantener el lote permite recuperar preparación y D1 ya completados.
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'secop_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra este notebook desde la carpeta del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from flows.alertas_flow import alertas_flow, configuration
from secop_pipeline.alertas import GOLD

LOTE = 'desde_gold_v2_20261009'
PUBLICAR_AWS = False
config = configuration(ROOT/'alertas_config.json')
if config.version != 'alertas-cd-2.0':
    raise ValueError('Se requiere modelo 2.0. Reinicie el kernel y revise la configuración.')
if not GOLD.is_file():
    raise FileNotFoundError(GOLD)

print('Fuente:', GOLD)
print('Lote:', LOTE, '| AWS:', PUBLICAR_AWS)
print('Se recuperarán las etapas válidas; siga el avance en el notebook 11.')
resultado = alertas_flow(
    config_path=str(ROOT/'alertas_config.json'),
    source=str(GOLD),
    cache_namespace=LOTE,
    publish_aws=PUBLICAR_AWS,
    only_load=False,
    scenarios=False,
)
resumen = resultado['resumen']
print('Estado:', resultado['estado'])
print('Resultados:', resultado['directorio'])
display(pd.DataFrame.from_dict(resumen['dimensiones'], orient='index'))


## 3. Resultado de cada dimensión

El porcentaje de activación usa como denominador los contratos **evaluados en esa dimensión**. La cobertura usa todo el universo. Esto impide presentar falta de información como ausencia de alerta.


In [ ]:
dimensiones = pd.DataFrame.from_dict(resumen['dimensiones'], orient='index')
dimensiones.index.name = 'Dimensión'
display(dimensiones[['alertas','sin_alerta','evaluados','no_evaluables','total',
                    'cobertura_pct','alertas_sobre_evaluados_pct','alertas_sobre_total_pct']])
for nombre, r in dimensiones.iterrows():
    tasa = 'no calculable' if pd.isna(r.alertas_sobre_evaluados_pct) else f'{r.alertas_sobre_evaluados_pct:.2f}%'
    print(f'{nombre}: {int(r.alertas):,} contratos con señal; activación {tasa}; {int(r.no_evaluables):,} no evaluables.')


## 4. Indicador compuesto y porcentaje de criterios activados

**Indicador:** suma de las señales observadas, de 0 a 3. Los NULL no aportan señales, pero se conservan como incertidumbre:

- `score_min` = señales observadas.
- `score_max` = señales observadas + dimensiones no evaluables.
- **Porcentaje mínimo de criterios activados** = 100 × score_min / 3.
- **Porcentaje máximo posible** = 100 × score_max / 3.
- **Cobertura** = 100 × dimensiones evaluadas / 3.

Aquí «cumplimiento» significa **activación de criterios de alerta**, no cumplimiento legal. Solo con cobertura del 100% el porcentaje es exacto. Ejemplo: [1, NULL, 0] produce indicador entre 1 y 2, porcentaje entre 33,33% y 66,67%, y cobertura 66,67%.

**Clasificación:** ALERTA si score_min ≥ umbral; NO_ALERTA si score_max < umbral; NO_EVALUABLE en los demás casos. Una clasificación puede ser concluyente aun con una dimensión desconocida. La ausencia de alerta no acredita legalidad.


In [ ]:
combinaciones = pd.DataFrame(resumen.get('combinaciones', []))
if not combinaciones.empty:
    flags = combinaciones[['D1','D2','D3']].apply(pd.to_numeric, errors='coerce')
    combinaciones['dimensiones_evaluadas'] = flags.notna().sum(axis=1)
    combinaciones['score_min'] = flags.fillna(0).sum(axis=1).astype(int)
    combinaciones['score_max'] = combinaciones.score_min + 3-combinaciones.dimensiones_evaluadas
    combinaciones['criterios_activados_min_pct'] = 100*combinaciones.score_min/3
    combinaciones['criterios_activados_max_pct'] = 100*combinaciones.score_max/3
    combinaciones['cobertura_pct'] = 100*combinaciones.dimensiones_evaluadas/3
    display(combinaciones)
clasificacion = pd.Series(resumen['clasificaciones'], name='contratos').to_frame()
total = resumen['contratos_resultado']
clasificacion['porcentaje_universo'] = 100*clasificacion.contratos/total if total else float('nan')
display(clasificacion)
print('Umbral del indicador:', config.score_threshold)
print('Contratos únicos con alguna señal:', resumen.get('contratos_con_alguna_senal'))
print('Señales sumadas (pueden coincidir):', resumen.get('total_senales'))
if total and not combinaciones.empty:
    cobertura = (combinaciones.dimensiones_evaluadas*combinaciones.contratos).sum()/(3*total)*100
    print(f'Cobertura global de evaluaciones contrato-dimensión: {cobertura:.2f}%.')
print('Continúe con notebook 10 para los agregados del mismo lote.')
